In [2]:
import os
import numpy as np
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
from scarches.models.scpoli import scPoli
from scarches.dataset.trvae.data_handling import remove_sparsity

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/torch/cuda/__init__.py:56: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel fro

In [67]:
source_adata = sc.read('/storage2/liuxiaodongLab/fanxueying/developmental_atlas/code/20251015_pretrain_pre_post_model/prepost_reference_model_lineage_20251103_v2/adata.h5ad')

In [68]:
brain_adata = sc.read('/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/filter_defined_number_combined_sampled.h5ad')
brain_adata

AnnData object with n_obs × n_vars = 437040 × 29772
    obs: 'stage', 'nCount_RNA', 'nFeature_RNA', 'orig_anno', 'orig_sub_anno', 'sample', 'percent.mt', 'reanno', 'orig.ident', 'lineage', 'dataset'
    obsm: 'X_umap'
    layers: 'counts'

In [51]:
source_adata

AnnData object with n_obs × n_vars = 36286 × 2000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'stage', 'percent.mt', 'species', 'platform', 'reanno', 'lineage', 'conditions_combined', 'embryo'
    layers: 'counts'

In [69]:
mask = brain_adata.obs['dataset'].isin(['fetal', 'adult'])

# Subset in place — modifies adata directly
brain_adata._inplace_subset_obs(mask)

In [6]:
print(brain_adata.obs['lineage'].value_counts())

lineage
Neuron            224098
Radial glia        38376
Neuroblast         38376
Neuronal IPC       27292
Oligo              12730
Astrocyte           8528
Vascular            8528
Fibroblast          8095
Immune              8017
Choroid plexus      4697
OPC                 4326
Ependymal           4264
Erythrocyte         4264
Glioblast           4264
Microglia           4264
Placodes             873
Neural crest         871
Name: count, dtype: int64


In [70]:
import scipy
from scipy import sparse
def remove_sparsity(adata):
    """
    If ``adata.X`` is a sparse matrix, this will convert it in to normal matrix.
    """
    if sparse.issparse(adata.X):
        adata.X = adata.X.toarray()
    return adata
brain_adata = remove_sparsity(brain_adata)

In [71]:
all_genes = source_adata.var_names
missing_genes = all_genes.difference(brain_adata.var_names)
missing_data = np.zeros((brain_adata.shape[0], len(missing_genes)))
query_adata_df = pd.DataFrame(brain_adata.X, columns=brain_adata.var_names, index=brain_adata.obs_names)
missing_df = pd.DataFrame(missing_data, columns=missing_genes, index=brain_adata.obs_names)
query_adata_combined_df = pd.concat([query_adata_df, missing_df], axis=1)[all_genes]
query_adata_extended = sc.AnnData(
    X=query_adata_combined_df.values,
    obs=brain_adata.obs,
    var=pd.DataFrame(index=all_genes),
    layers={'counts': query_adata_combined_df.values}
)

In [72]:
obs_to_keep = ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'stage', 'percent.mt', 
               'platform', 'species', 'embryo', 'lineage', 
                'reanno', 'dataset']
layers_to_keep = ['counts', 'logcounts']

# Check which columns and layers actually exist in the dataset
existing_obs = [col for col in obs_to_keep if col in query_adata_extended.obs.columns]
existing_layers = [layer for layer in layers_to_keep if layer in query_adata_extended.layers.keys()]

# Create a simplified AnnData object
query_adata_clean = sc.AnnData(
    X=query_adata_extended.X.copy(),
    obs=query_adata_extended.obs[existing_obs].copy(),
    var=pd.DataFrame(index=query_adata_extended.var.index),  # Keep only gene index
    layers={layer: query_adata_extended.layers[layer].copy() for layer in existing_layers}
)

# Replace original object with the cleaned version
query_adata_extended = query_adata_clean

print("Cleaned adata:")
print(query_adata_extended)

Cleaned adata:
AnnData object with n_obs × n_vars = 401863 × 2000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'stage', 'percent.mt', 'lineage', 'reanno', 'dataset'
    layers: 'counts'


In [73]:
query_adata_extended.X = query_adata_extended.layers["counts"]
source_adata.X = source_adata.layers["counts"]

In [74]:
# --- Check for duplicate cell names (obs index) ---
source_cell_names = set(source_adata.obs.index)
query_cell_names = set(query_adata_extended.obs.index)
duplicate_cell_names = source_cell_names.intersection(query_cell_names)

if duplicate_cell_names:
    print(f"Found {len(duplicate_cell_names)} duplicate cell names. These will be overwritten by query data.")
    # Option 1: Remove duplicates from source_adata
    source_adata_unique = source_adata[~source_adata.obs.index.isin(duplicate_cell_names)]
    print(f"Removed {len(duplicate_cell_names)} duplicate cells from source data.")
    # Use the deduplicated source data for concatenation
    combined_adata = sc.concat([source_adata_unique, query_adata_extended], axis=0, join="outer")
else:
    print("No duplicate cell names found.")
    # If no duplicates, proceed with standard concatenation
    combined_adata = sc.concat([source_adata, query_adata_extended], axis=0, join="outer")

# Ensure the combined data is not sparse if required by downstream functions
combined_adata = remove_sparsity(combined_adata)

print(f"Shape of combined AnnData: {combined_adata.shape}")
print(f"Columns in .obs: {list(combined_adata.obs.columns)}")

No duplicate cell names found.
Shape of combined AnnData: (438149, 2000)
Columns in .obs: ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'stage', 'percent.mt', 'species', 'platform', 'reanno', 'lineage', 'conditions_combined', 'embryo', 'dataset']


In [16]:
print(combined_adata.obs["lineage"].value_counts())

lineage
Neuron              224098
Radial glia          38376
Neuroblast           38376
Neuronal IPC         27292
TE_TrB               15935
Oligo                12730
meso_Exe.meso        11466
Vascular              8528
Astrocyte             8528
Fibroblast            8095
Immune                8017
Choroid plexus        4697
OPC                   4326
Glioblast             4264
Microglia             4264
Erythrocyte           4264
Ependymal             4264
epi                   2540
neural_ecto           2001
hemogenic             1009
ExE_endo               955
Placodes               873
Neural crest           871
NMP                    812
Amniotic_ecto          521
Primitive.streak       367
PGC                    210
Endoderm               203
Notochord               91
Inner Cell Mass         88
Prelineage              39
8C                      30
Morula                  19
Name: count, dtype: int64


In [75]:
combined_adata.layers['counts'] = combined_adata.X.copy()

In [20]:
combined_adata.obs

,orig.ident,nCount_RNA,nFeature_RNA,stage,percent.mt,species,platform,reanno,lineage,conditions_combined,embryo,dataset
AAACCTGAGACGCTTT-1_1_1_1_1_1_1,mole,112028.0,9381,E9_IVC,5.228157,Homo sapiens,10x-Genomics,EVT_1,TE_TrB,mole,NaN,NaN
AACGTTGAGACGCAAC-1_1_1_1_1_1_1,mole,191885.0,8618,E9_IVC,14.688485,Homo sapiens,10x-Genomics,STB_3,TE_TrB,mole,NaN,NaN
AACTCAGCAGACGCCT-1_1_1_1_1_1_1,mole,189565.0,7942,E9_IVC,7.517738,Homo sapiens,10x-Genomics,STB_2,TE_TrB,mole,NaN,NaN
AAGACCTTCGGCGCTA-1_1_1_1_1_1_1,mole,25545.0,4869,E9_IVC,17.987865,Homo sapiens,10x-Genomics,CTB_2,TE_TrB,mole,NaN,NaN
AAGCCGCCATATGGTC-1_1_1_1_1_1_1,mole,107601.0,10074,E9_IVC,3.851265,Homo sapiens,10x-Genomics,Hypoblast_2,ExE_endo,mole,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
10X213_4:CTGCATCCATGTCTAG,Braun,33372.0,6696.0,13.0,0.046146,NaN,NaN,Vascular,Vascular,NaN,NaN,fetal
10X254_5:TGTTCCGGTTGTCAGT,Braun,5217.0,2667.0,14.0,0.073414,NaN,NaN,Vascular,Vascular,NaN,NaN,fetal
10X255_7:TTATTGCAGGTGTGAC,Braun,34965.0,6990.0,14.0,0.042957,NaN,NaN,Vascular,Vascular,NaN,NaN,fetal
10X114_3:TCAATCTGTGTCCTCT,Braun,2323.0,1292.0,9.199999809265137,0.029272,NaN,NaN,Vascular,Vascular,NaN,NaN,fetal


In [76]:
# 确保数值列是数值类型
numeric_columns = ['nCount_RNA', 'nFeature_RNA', 'percent.mt']

for col in numeric_columns:
    if col in combined_adata.obs.columns:
        combined_adata.obs[col] = pd.to_numeric(combined_adata.obs[col], errors='coerce')
        print(f"已将列 '{col}' 转换为数值类型，数据类型: {combined_adata.obs[col].dtype}")

# 现在写入文件
combined_adata.write_h5ad("/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/embryo_hvg2000_merge_fetal_adult.h5ad")

已将列 'nCount_RNA' 转换为数值类型，数据类型: float64
已将列 'nFeature_RNA' 转换为数值类型，数据类型: float64
已将列 'percent.mt' 转换为数值类型，数据类型: float64


In [84]:
brain_adata = sc.read('/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/filter_defined_number_combined_sampled.h5ad')

In [26]:
brain_adata

AnnData object with n_obs × n_vars = 437040 × 29772
    obs: 'stage', 'nCount_RNA', 'nFeature_RNA', 'orig_anno', 'orig_sub_anno', 'sample', 'percent.mt', 'reanno', 'orig.ident', 'lineage', 'dataset'
    obsm: 'X_umap'
    layers: 'counts'

In [85]:
pre_fetal = brain_adata[brain_adata.obs['dataset'].isin(['fetal', 'pre'])]

In [30]:
pre_fetal

View of AnnData object with n_obs × n_vars = 204925 × 29772
    obs: 'stage', 'nCount_RNA', 'nFeature_RNA', 'orig_anno', 'orig_sub_anno', 'sample', 'percent.mt', 'reanno', 'orig.ident', 'lineage', 'dataset'
    obsm: 'X_umap'
    layers: 'counts'

In [86]:
adata_tmp = pre_fetal.copy()

# 对临时数据进行标准化和log转换
sc.pp.normalize_total(adata_tmp, target_sum=1e4)
sc.pp.log1p(adata_tmp)

# 选择高变基因
sc.pp.highly_variable_genes(
    adata_tmp,
    n_top_genes=4000,
    flavor="cell_ranger",
    batch_key="orig.ident",
    subset=False
)

# 使用高变基因筛选原始数据
hvg_mask = adata_tmp.var['highly_variable']
adata_hvg = pre_fetal[:, hvg_mask].copy()


In [87]:
if hasattr(adata_hvg.X, 'toarray'):
    # 如果是稀疏矩阵，转换为稠密矩阵
    adata_hvg.X = adata_hvg.X.toarray()
    print("已将稀疏矩阵转换为稠密矩阵")

# 确保counts层存在
if "counts" not in adata_hvg.layers:
    adata_hvg.layers["counts"] = adata_hvg.X.copy()
    print("创建了counts层")
else:
    # 如果counts层存在但也是稀疏矩阵，同样需要转换
    if hasattr(adata_hvg.layers["counts"], 'toarray'):
        adata_hvg.layers["counts"] = adata_hvg.layers["counts"].toarray()
    adata_hvg.X = adata_hvg.layers["counts"].copy()


已将稀疏矩阵转换为稠密矩阵


In [89]:
adata_hvg

AnnData object with n_obs × n_vars = 204925 × 4000
    obs: 'stage', 'nCount_RNA', 'nFeature_RNA', 'orig_anno', 'orig_sub_anno', 'sample', 'percent.mt', 'reanno', 'orig.ident', 'lineage', 'dataset'
    obsm: 'X_umap'
    layers: 'counts'

In [88]:
adata_hvg.write_h5ad("/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/embryo_fetal_hvg4000.h5ad")

In [65]:
# 确保数值列是数值类型
numeric_columns = ['nCount_RNA', 'nFeature_RNA', 'percent.mt']

for col in numeric_columns:
    if col in combined_adata.obs.columns:
        combined_adata.obs[col] = pd.to_numeric(combined_adata.obs[col], errors='coerce')
        print(f"已将列 '{col}' 转换为数值类型，数据类型: {combined_adata.obs[col].dtype}")

# 现在写入文件
combined_adata.write_h5ad("/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/embryo_hvg2000_merge_adult.h5ad")

AnnData object with n_obs × n_vars = 204925 × 2000
    obs: 'stage', 'nCount_RNA', 'nFeature_RNA', 'orig_anno', 'orig_sub_anno', 'sample', 'percent.mt', 'reanno', 'orig.ident', 'lineage', 'dataset'
    obsm: 'X_umap'
    layers: 'counts'

In [3]:
pre_fetal=sc.read_h5ad("/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/model_defined_number/without_invitro_embryo_fetal_hvg2000_dims50/scpoli_model_lineage_hvg2000/adata.h5ad")

In [4]:
brain_adata = sc.read('/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/filter_defined_number_combined_sampled.h5ad')

In [9]:
adult= brain_adata[brain_adata.obs['dataset']=="adult"]

In [13]:
missing_genes

CategoricalIndex([], categories=['5S_rRNA', '5_8S_rRNA', '7SK', 'A1BG', ..., 'hsa-mir-1253', 'hsa-mir-423', 'hsa-mir-8069-1', 'snoZ196'], ordered=False, dtype='category')

In [17]:
all_genes = pre_fetal.var_names
missing_genes = all_genes.difference(adult.var_names)
missing_data = np.zeros((adult.shape[0], len(missing_genes)))

# 修复：将稀疏矩阵转换为稠密矩阵
query_adata_df = pd.DataFrame(adult.X.toarray(), columns=adult.var_names, index=adult.obs_names)

missing_df = pd.DataFrame(missing_data, columns=missing_genes, index=adult.obs_names)
query_adata_combined_df = pd.concat([query_adata_df, missing_df], axis=1)[all_genes]
query_adata_extended = sc.AnnData(
    X=query_adata_combined_df.values,
    obs=adult.obs,
    var=pd.DataFrame(index=all_genes),
    layers={'counts': query_adata_combined_df.values}
)

In [18]:
obs_to_keep = ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'stage', 'percent.mt', 
               'platform', 'species', 'embryo', 'lineage', 
                'reanno', 'dataset']
layers_to_keep = ['counts', 'logcounts']

# Check which columns and layers actually exist in the dataset
existing_obs = [col for col in obs_to_keep if col in query_adata_extended.obs.columns]
existing_layers = [layer for layer in layers_to_keep if layer in query_adata_extended.layers.keys()]

# Create a simplified AnnData object
query_adata_clean = sc.AnnData(
    X=query_adata_extended.X.copy(),
    obs=query_adata_extended.obs[existing_obs].copy(),
    var=pd.DataFrame(index=query_adata_extended.var.index),  # Keep only gene index
    layers={layer: query_adata_extended.layers[layer].copy() for layer in existing_layers}
)

# Replace original object with the cleaned version
query_adata_extended = query_adata_clean

print("Cleaned adata:")
print(query_adata_extended)

Cleaned adata:
AnnData object with n_obs × n_vars = 232115 × 2000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'stage', 'percent.mt', 'lineage', 'reanno', 'dataset'
    layers: 'counts'


In [20]:
query_adata_extended.X = query_adata_extended.layers["counts"]
pre_fetal.X = pre_fetal.layers["counts"]

In [21]:
# --- Check for duplicate cell names (obs index) ---
source_cell_names = set(pre_fetal.obs.index)
query_cell_names = set(query_adata_extended.obs.index)
duplicate_cell_names = source_cell_names.intersection(query_cell_names)

if duplicate_cell_names:
    print(f"Found {len(duplicate_cell_names)} duplicate cell names. These will be overwritten by query data.")
    # Option 1: Remove duplicates from source_adata
    source_adata_unique = pre_fetal[~pre_fetal.obs.index.isin(duplicate_cell_names)]
    print(f"Removed {len(duplicate_cell_names)} duplicate cells from source data.")
    # Use the deduplicated source data for concatenation
    combined_adata = sc.concat([source_adata_unique, query_adata_extended], axis=0, join="outer")
else:
    print("No duplicate cell names found.")
    # If no duplicates, proceed with standard concatenation
    combined_adata = sc.concat([pre_fetal, query_adata_extended], axis=0, join="outer")

# Ensure the combined data is not sparse if required by downstream functions
combined_adata = remove_sparsity(combined_adata)

print(f"Shape of combined AnnData: {combined_adata.shape}")
print(f"Columns in .obs: {list(combined_adata.obs.columns)}")

No duplicate cell names found.
Shape of combined AnnData: (437040, 2000)
Columns in .obs: ['stage', 'nCount_RNA', 'nFeature_RNA', 'orig_anno', 'orig_sub_anno', 'sample', 'percent.mt', 'reanno', 'orig.ident', 'lineage', 'dataset', 'conditions_combined']


In [22]:
combined_adata.layers['counts'] = combined_adata.X.copy()

In [25]:
combined_adata

AnnData object with n_obs × n_vars = 437040 × 2000
    obs: 'stage', 'nCount_RNA', 'nFeature_RNA', 'orig_anno', 'orig_sub_anno', 'sample', 'percent.mt', 'reanno', 'orig.ident', 'lineage', 'dataset', 'conditions_combined'
    obsm: 'X_umap'
    layers: 'counts'

In [26]:
# 确保数值列是数值类型
numeric_columns = ['nCount_RNA', 'nFeature_RNA', 'percent.mt']

for col in numeric_columns:
    if col in combined_adata.obs.columns:
        combined_adata.obs[col] = pd.to_numeric(combined_adata.obs[col], errors='coerce')
        print(f"已将列 '{col}' 转换为数值类型，数据类型: {combined_adata.obs[col].dtype}")

# 现在写入文件
combined_adata.write_h5ad("/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/embryo_fetal_hvg2000_merge_adult.h5ad")

已将列 'nCount_RNA' 转换为数值类型，数据类型: float64
已将列 'nFeature_RNA' 转换为数值类型，数据类型: float64
已将列 'percent.mt' 转换为数值类型，数据类型: float64
